Load config from yaml

In [1]:
import yaml
from nam.utils.config import NAMConfig

def load_config(path: str) -> NAMConfig:
    with open(path) as f:
        raw = yaml.safe_load(f)
    return NAMConfig(**raw)

config = load_config(r"C:\Users\maart\OneDrive\Documenten\Universiteit\Scriptie\python_repo\thesis-nam\configs\compas_baseline.yaml")


Process data and create datasets

In [2]:
from nam.models.nam import NAM
from nam.models.feature_nn import FeatureNN
from nam.data.dataset import NAMDataset
from nam.training.losses import penalized_loss
from nam.data.data_utils import load_compas, preprocess, split

df = load_compas(r"C:\Users\maart\OneDrive\Documenten\Universiteit\Scriptie\python_repo\thesis-nam\datasets\raw\compas-scores-two-years.csv")
X,y,feature_names = preprocess(df)

X_train,X_val,X_test,y_train,y_val,y_test = split(X,y,config.val_frac,config.test_frac,config.seed)

train_dataset = NAMDataset(X_train, y_train)
val_dataset   = NAMDataset(X_val,   y_val)
test_dataset  = NAMDataset(X_test,  y_test)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")
print(f"Features: {len(feature_names)} → {feature_names}")

# Check one sample looks right
features, target, weight = train_dataset[0]
print(features.shape, target, weight)

#model = NAM()

Train: 4834, Val: 1036, Test: 1037
Features: 13 → ['age', 'priors_count', 'length_of_stay', 'c_charge_degree_F', 'c_charge_degree_M', 'race_African-American', 'race_Asian', 'race_Caucasian', 'race_Hispanic', 'race_Native American', 'race_Other', 'sex_Female', 'sex_Male']
torch.Size([13]) tensor(0.) tensor(1.)


DataLoaders:

In [3]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset,config.batch_size,shuffle=True)
val_loader = DataLoader(val_dataset,config.batch_size,shuffle=True)
test_loader =  DataLoader(test_dataset,config.batch_size,shuffle=True)

Forward pass

In [ ]:
from nam.training.losses import penalized_loss
from nam.training.metrics import auroc
import torch
import copy
import random
import numpy as np

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(config.seed)

num_features = X_train.shape[1]

model = NAM(
    num_features=num_features,
    num_units=config.num_units,
    hidden_sizes=config.hidden_sizes,
    dropout=config.dropout,
    feature_dropout=config.feature_dropout,
    activation=config.activation
)

optimizer = torch.optim.Adam(model.parameters(),lr = config.lr)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=config.decay_rate)

best_auc =0.0
best_model_state = copy.deepcopy(model.state_dict())

for epoch in range(1000):
    model.train()
    epoch_loss = 0
    for X_batch, y_batch, weights in train_loader:
        optimizer.zero_grad()
        predictions, fnn_outputs = model(X_batch)
        loss = penalized_loss(logits=predictions,
                            targets=y_batch,
                            weights=weights,
                            fnn_outputs=fnn_outputs,
                            model = model,
                            output_regularization=config.output_regularization,
                            l2_regularization=config.l2_regularization,
                            task = config.task)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    model.eval()
    all_predictions = []
    all_targets = []
    with torch.no_grad():
        for X_val_batch, y_val_batch, _ in val_loader:
            val_predictions, val_fnn_outputs = model(X_val_batch)
            all_predictions.append(val_predictions)
            all_targets.append(y_val_batch)

    val_predictions = torch.cat(all_predictions)
    val_targets = torch.cat(all_targets)
    auc = auroc(val_predictions,val_targets)

    if auc>best_auc:
        best_auc = auc
        best_model_state = copy.deepcopy(model.state_dict())

    scheduler.step()

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1}/1000 | loss={epoch_loss/len(train_loader):.4f} | val_auc={auc:.4f} | best={best_auc:.4f}")

#Final test performance:
model.load_state_dict(best_model_state)
model.eval()
all_predictions = []
all_targets = []

with torch.no_grad():
    for X_test_batch, y_test_batch, _ in test_loader:
        test_predictions, test_fnn_outputs = model(X_test_batch)
        all_predictions.append(test_predictions)
        all_targets.append(y_test_batch)

    test_predictions = torch.cat(all_predictions)
    test_targets = torch.cat(all_targets)
    final_auc = auroc(test_predictions,test_targets)

print(final_auc)
        
